In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv(r"C:\Users\USER\Desktop\碩論\程式碼\embedding_data.csv")

data

,X2,X3,X1,Y
0,255.000000,94.940954,162.097863,4.686122
1,215.309400,130.434396,131.702366,4.410815
2,188.692556,112.414072,176.294826,3.444338
3,197.089163,155.840962,81.755694,3.378867
4,208.651847,70.102607,70.191431,3.135581
...,...,...,...,...
95,47.862764,55.539969,181.891681,-2.831205
96,56.421777,89.129206,191.903929,-3.037451
97,60.563777,24.069923,0.000000,-3.492096
98,86.754581,59.616503,135.187503,-3.672475


In [2]:
X = data.iloc[:,:-1]
y = data["Y"]

print(y.head())

0    4.686122
1    4.410815
2    3.444338
3    3.378867
4    3.135581
Name: Y, dtype: float64


In [3]:
import torch

X_tensor = torch.tensor(
    X.values,
    dtype=torch.float32
)

y_tensor = torch.tensor(
    y.values,
    dtype=torch.float32
)

In [4]:
y_tensor = y_tensor.unsqueeze(1)

In [5]:
import torch.nn as nn

class FeatureNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(3,16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16,3)

    def forward(self, x):

        delta = self.fc1(x)

        delta = self.relu(delta)

        delta = self.fc2(delta)

        z = x + delta

        return z, delta

In [6]:
class GLMLayer(nn.Module):

    def __init__(self):

        super().__init__()

        self.linear = nn.Linear(3,1)

    def forward(self,z):

        return self.linear(z)

In [7]:
feature_net = FeatureNet()

glm_layer = GLMLayer()

In [8]:
z, delta = feature_net(X_tensor)

y_hat = glm_layer(z)

In [9]:
print(z.shape)
print(delta.shape)

torch.Size([100, 3])
torch.Size([100, 3])


In [10]:
print(delta[:5])

tensor([[101.6481, -76.6330,  30.5156],
        [ 81.9273, -71.4062,  12.4678],
        [ 80.7508, -60.4458,  31.8826],
        [ 67.0793, -71.3128, -10.0295],
        [ 74.5768, -63.6884,   5.7791]], grad_fn=<SliceBackward0>)


In [11]:
import torch.nn as nn

criterion = nn.MSELoss()

In [12]:
import torch.optim as optim

model_params = list(feature_net.parameters()) + list(glm_layer.parameters())

optimizer = optim.Adam(model_params, lr=1e-5)

In [19]:
epochs = 1000
lambda_reg = 0.01

loss_history = []
mse_history = []
corr_history = []

for epoch in range(epochs):

    # ---------- Forward ----------
    z, delta = feature_net(X_tensor)

    y_hat = glm_layer(z)

    # ---------- Loss ----------
    prediction_loss = criterion(y_hat, y_tensor)

    correction_loss = torch.mean(delta**2)

    loss = prediction_loss + lambda_reg * correction_loss

    # loss = prediction_loss

    # ---------- Backpropagation ----------
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    # ------------存下模型的學習軌跡------------
    if epoch % 1 == 0:  # 建議先全部存，之後再決定畫不畫
        loss_history.append(loss.item())
        mse_history.append(prediction_loss.item())
        corr_history.append(correction_loss.item())

    # ---------- Print ----------
    if epoch % 200 == 0:

        print(
            f"Epoch {epoch:4d} | "
            f"Total={loss.item():.4f} | "
            f"MSE={prediction_loss.item():.4f} | "
            f"Correction={correction_loss.item():.4f}"
        )

Epoch    0 | Total=3079.5706 | MSE=3060.8694 | Correction=1870.1261
Epoch  200 | Total=2910.2573 | MSE=2890.8562 | Correction=1940.1176
Epoch  400 | Total=2749.1147 | MSE=2728.9363 | Correction=2017.8398
Epoch  600 | Total=2595.6531 | MSE=2574.6189 | Correction=2103.4092
Epoch  800 | Total=2449.6140 | MSE=2427.6494 | Correction=2196.4541


In [20]:
len(loss_history)

1000

In [21]:
print(delta.mean(dim=0))

print(delta.std(dim=0))

print(delta.min(dim=0).values)

print(delta.max(dim=0).values)

tensor([ 22.9193, -71.1395, -21.2228], grad_fn=<MeanBackward1>)
tensor([11.3021, 20.9490, 17.1729], grad_fn=<StdBackward0>)
tensor([   3.6854, -126.2485,  -63.7158], grad_fn=<MinBackward0>)
tensor([ 60.3597, -19.6873,  19.9437], grad_fn=<MaxBackward0>)


In [22]:
print(X_tensor[:10])
print(z[:10])

tensor([[255.0000,  94.9410, 162.0979],
        [215.3094, 130.4344, 131.7024],
        [188.6926, 112.4141, 176.2948],
        [197.0892, 155.8410,  81.7557],
        [208.6518,  70.1026,  70.1914],
        [149.9482, 123.8316,  94.8743],
        [116.4670, 137.5962, 164.5834],
        [181.6945, 236.9369, 154.7280],
        [129.6991, 162.2985, 176.3539],
        [173.7452, 141.9908, 150.0218]])
tensor([[315.3597, -31.3075, 135.3373],
        [260.8540,  14.8699,  92.9011],
        [232.9314,   7.6781, 156.7703],
        [232.4672,  45.7418,  26.4898],
        [254.5454, -27.9423,  36.3854],
        [178.5430,  37.0271,  58.7645],
        [140.8838,  57.1181, 145.2578],
        [211.1418, 114.3734,  95.3582],
        [155.6442,  71.6252, 150.7299],
        [209.7647,  39.2136, 117.7229]], grad_fn=<SliceBackward0>)
